In [1]:
%pip install psycopg2-binary
from pathlib import Path
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

Note: you may need to restart the kernel to use updated packages.


In [2]:
data_dirs = [
    Path("../data/Raw/mimic-iv-clinical-database-demo-2.2/hosp"),
    Path("../data/Raw/mimic-iv-clinical-database-demo-2.2/icu")
]

dfs = {}

for data_dir in data_dirs:
    for file in data_dir.glob("*.csv.gz"):
        name = file.name.replace(".csv.gz", "")
        dfs[name] = pd.read_csv(file)

print(dfs.keys())

C:\Users\Dell\AppData\Local\Temp\ipykernel_1188\3010911833.py:11: DtypeWarning: Columns (4,6,7,8,9,10,11,12,13,15,16,17,18,21,23,24,25,26,27,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[name] = pd.read_csv(file)


dict_keys(['admissions', 'diagnoses_icd', 'drgcodes', 'd_hcpcs', 'd_icd_diagnoses', 'd_icd_procedures', 'd_labitems', 'emar', 'emar_detail', 'hcpcsevents', 'labevents', 'microbiologyevents', 'omr', 'patients', 'pharmacy', 'poe', 'poe_detail', 'prescriptions', 'procedures_icd', 'provider', 'services', 'transfers', 'caregiver', 'chartevents', 'datetimeevents', 'd_items', 'icustays', 'ingredientevents', 'inputevents', 'outputevents', 'procedureevents'])


In [3]:
dfs["admissions"].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 275 entries, 0 to 274
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   subject_id            275 non-null    int64 
 1   hadm_id               275 non-null    int64 
 2   admittime             275 non-null    object
 3   dischtime             275 non-null    object
 4   deathtime             15 non-null     object
 5   admission_type        275 non-null    object
 6   admit_provider_id     275 non-null    object
 7   admission_location    275 non-null    object
 8   discharge_location    233 non-null    object
 9   insurance             275 non-null    object
 10  language              275 non-null    object
 11  marital_status        263 non-null    object
 12  race                  275 non-null    object
 13  edregtime             182 non-null    object
 14  edouttime             182 non-null    object
 15  hospital_expire_flag  275 non-null    in

In [4]:
load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
with engine.connect() as connection:
    print("PostgreSQL connection successful.")

PostgreSQL connection successful.


In [5]:
for table_name, df in dfs.items():
    print(f"Loading {table_name}...")

    df.to_sql(
        table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=5000
    )

    print(f"✓ {table_name} loaded")

Loading admissions...
✓ admissions loaded
Loading diagnoses_icd...
✓ diagnoses_icd loaded
Loading drgcodes...
✓ drgcodes loaded
Loading d_hcpcs...
✓ d_hcpcs loaded
Loading d_icd_diagnoses...
✓ d_icd_diagnoses loaded
Loading d_icd_procedures...
✓ d_icd_procedures loaded
Loading d_labitems...
✓ d_labitems loaded
Loading emar...
✓ emar loaded
Loading emar_detail...
✓ emar_detail loaded
Loading hcpcsevents...
✓ hcpcsevents loaded
Loading labevents...
✓ labevents loaded
Loading microbiologyevents...
✓ microbiologyevents loaded
Loading omr...
✓ omr loaded
Loading patients...
✓ patients loaded
Loading pharmacy...
✓ pharmacy loaded
Loading poe...
✓ poe loaded
Loading poe_detail...
✓ poe_detail loaded
Loading prescriptions...
✓ prescriptions loaded
Loading procedures_icd...
✓ procedures_icd loaded
Loading provider...
✓ provider loaded
Loading services...
✓ services loaded
Loading transfers...
✓ transfers loaded
Loading caregiver...
✓ caregiver loaded
Loading chartevents...
✓ chartevents loaded


In [6]:
tables = pd.read_sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""", engine)

print(tables)

            table_name
0           admissions
1            caregiver
2          chartevents
3              d_hcpcs
4      d_icd_diagnoses
5     d_icd_procedures
6              d_items
7           d_labitems
8       datetimeevents
9        diagnoses_icd
10            drgcodes
11                emar
12         emar_detail
13         hcpcsevents
14            icustays
15    ingredientevents
16         inputevents
17           labevents
18  microbiologyevents
19                 omr
20        outputevents
21            patients
22            pharmacy
23                 poe
24          poe_detail
25       prescriptions
26     procedureevents
27      procedures_icd
28            provider
29            services
30           transfers


In [7]:
for table in tables["table_name"]:
    count = pd.read_sql(
        f'SELECT COUNT(*) AS rows FROM "{table}";',
        engine
    ).iloc[0, 0]

    print(f"{table:25} {count:,} rows")

admissions                275 rows
caregiver                 15,468 rows
chartevents               668,862 rows
d_hcpcs                   89,200 rows
d_icd_diagnoses           109,775 rows
d_icd_procedures          85,257 rows
d_items                   4,014 rows
d_labitems                1,622 rows
datetimeevents            15,280 rows
diagnoses_icd             4,506 rows
drgcodes                  454 rows
emar                      35,835 rows
emar_detail               72,018 rows
hcpcsevents               61 rows
icustays                  140 rows
ingredientevents          25,728 rows
inputevents               20,404 rows
labevents                 107,727 rows
microbiologyevents        2,899 rows
omr                       2,964 rows
outputevents              9,362 rows
patients                  100 rows
pharmacy                  15,306 rows
poe                       45,154 rows
poe_detail                3,795 rows
prescriptions             18,087 rows
procedureevents           1,468 

In [12]:
count = pd.read_sql(
        f'SELECT * FROM  patients',
        engine
    )

In [13]:
count

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10014729,F,21,2125,2011 - 2013,None
1,10003400,F,72,2134,2011 - 2013,2137-09-02
2,10002428,F,80,2155,2011 - 2013,None
3,10032725,F,38,2143,2011 - 2013,2143-03-30
4,10027445,F,48,2142,2011 - 2013,2146-02-09
...,...,...,...,...,...,...
95,10004733,M,51,2174,2014 - 2016,None
96,10021118,M,62,2161,2014 - 2016,None
97,10018501,M,83,2141,2014 - 2016,None
98,10007058,M,48,2167,2014 - 2016,None


In [14]:
count = pd.read_sql(
        f'SELECT * FROM  admissions',
        engine
    )
count

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,None,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0
1,10009628,25926192,2153-09-17 17:08:00,2153-09-25 13:20:00,None,URGENT,P41R5N,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,?,MARRIED,HISPANIC/LATINO - PUERTO RICAN,None,None,0
2,10018081,23983182,2134-08-18 02:02:00,2134-08-23 19:35:00,None,URGENT,P233F6,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicare,ENGLISH,MARRIED,WHITE,2134-08-17 16:24:00,2134-08-18 03:15:00,0
3,10006053,22942076,2111-11-13 23:39:00,2111-11-15 17:20:00,2111-11-15 17:20:00,URGENT,P38TI6,TRANSFER FROM HOSPITAL,DIED,Medicaid,ENGLISH,None,UNKNOWN,None,None,1
4,10031404,21606243,2113-08-04 18:46:00,2113-08-06 20:57:00,None,URGENT,P07HDB,TRANSFER FROM HOSPITAL,HOME,Other,ENGLISH,WIDOWED,WHITE,None,None,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,10038992,24745425,2187-07-29 01:05:00,2187-08-03 17:02:00,None,SURGICAL SAME DAY ADMISSION,P41R5N,PHYSICIAN REFERRAL,SKILLED NURSING FACILITY,Medicare,ENGLISH,MARRIED,WHITE,None,None,0
271,10008287,22168393,2145-09-28 01:17:00,2145-10-02 13:35:00,None,SURGICAL SAME DAY ADMISSION,P898NM,PHYSICIAN REFERRAL,HOME HEALTH CARE,Other,ENGLISH,SINGLE,WHITE,None,None,0
272,10022880,27708593,2177-03-12 07:15:00,2177-03-19 14:25:00,None,SURGICAL SAME DAY ADMISSION,P99698,PHYSICIAN REFERRAL,HOME,Medicare,ENGLISH,MARRIED,WHITE,None,None,0
273,10004457,23251352,2141-12-17 11:00:00,2141-12-21 15:56:00,None,SURGICAL SAME DAY ADMISSION,P41R5N,PHYSICIAN REFERRAL,REHAB,Medicare,ENGLISH,SINGLE,OTHER,None,None,0


In [15]:
count = pd.read_sql(
        f'SELECT * FROM  icustays',
        engine
    )
count

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10018328,23786647,31269608,Neuro Stepdown,Neuro Stepdown,2154-04-24 23:03:44,2154-05-02 15:55:21,7.702512
1,10020187,24104168,37509585,Neuro Surgical Intensive Care Unit (Neuro SICU),Neuro Stepdown,2169-01-15 04:56:00,2169-01-20 15:47:50,5.452662
2,10020187,26842957,32554129,Neuro Intermediate,Neuro Intermediate,2170-02-24 18:18:46,2170-02-25 15:15:26,0.872685
3,10012853,27882036,31338022,Trauma SICU (TSICU),Trauma SICU (TSICU),2176-11-26 02:34:49,2176-11-29 20:58:54,3.766725
4,10020740,25826145,32145159,Trauma SICU (TSICU),Trauma SICU (TSICU),2150-06-03 20:12:32,2150-06-04 21:05:58,1.037106
...,...,...,...,...,...,...,...,...
135,10020786,23488445,33683112,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2189-06-09 12:46:30,2189-06-10 22:58:09,1.424757
136,10020740,23831430,35026312,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2150-03-11 15:34:56,2150-03-19 02:17:47,7.446424
137,10032725,20611640,30101877,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2143-03-22 06:42:00,2143-03-25 15:05:33,3.349687
138,10037928,24656677,39804682,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2178-12-21 06:05:18,2178-12-22 02:16:08,0.840856


In [20]:
count = pd.read_sql(
        f'SELECT * FROM  d_icd_diagnoses',
        engine
    )
count


,icd_code,icd_version,long_title
0,0090,9,"Infectious colitis, enteritis, and gastroenter..."
1,01160,9,"Tuberculous pneumonia [any form], unspecified"
2,01186,9,"Other specified pulmonary tuberculosis, tuberc..."
3,01200,9,"Tuberculous pleurisy, unspecified"
4,01236,9,"Tuberculous laryngitis, tubercle bacilli not f..."
...,...,...,...
109770,Z88,10,"Allergy status to drugs, medicaments and biolo..."
109771,Z89012,10,Acquired absence of left thumb
109772,Z90410,10,Acquired total absence of pancreas
109773,Z948,10,Other transplanted organ and tissue status


In [22]:
count = pd.read_sql(
        f'SELECT * FROM  diagnoses_icd',
        engine
    )
count

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10035185,22580999,3,4139,9
1,10035185,22580999,10,V707,9
2,10035185,22580999,1,41401,9
3,10035185,22580999,9,3899,9
4,10035185,22580999,11,V8532,9
...,...,...,...,...,...
4501,10004733,27411876,19,3129,9
4502,10004733,27411876,30,30000,9
4503,10004733,27411876,26,4739,9
4504,10004733,27411876,22,56210,9
